# Basic workflow for making a "surrogate" model that learns the unpolarized cross-section as function of $x_{\text{B}}$, $t$, $Q^{2}$, and $\phi$.

## (1): Import Libraries

In [ ]:
import datetime
import gc

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import regularizers
from sklearn.model_selection import train_test_split

## (2): Plotting Styles:

In [ ]:
plt.rcParams.update({
    "text.usetex": True, "font.family": "serif",
})
plt.rcParams['xtick.direction'] = 'in'
plt.rcParams['xtick.major.size'] = 8.5
plt.rcParams['xtick.major.width'] = 0.5
plt.rcParams['xtick.minor.size'] = 2.5
plt.rcParams['xtick.minor.width'] = 0.5
plt.rcParams['xtick.minor.visible'] = True
plt.rcParams['xtick.top'] = True
plt.rcParams['ytick.direction'] = 'in'
plt.rcParams['ytick.major.size'] = 8.5
plt.rcParams['ytick.major.width'] = 0.5
plt.rcParams['ytick.minor.size'] = 2.5
plt.rcParams['ytick.minor.width'] = 0.5
plt.rcParams['ytick.minor.visible'] = True
plt.rcParams['ytick.right'] = True
plt.rcParams['savefig.dpi'] = 300

## (3): Data Loading:

### (3.1): Loading Main File:

In [ ]:
test_dataframe = pd.read_csv('../data/burner_data.csv')

### (3.2): Loading in the supervised learning $(x, y)$ pairs:

In [ ]:
x_data = test_dataframe[["t", "x_b", "q_squared", "phi"]]
y_data = test_dataframe[["unp_beam_unp_target_xsec"]]

### (3.3): Checking out the $x$ data:

In [ ]:
x_data.head()

### (3.4): Checking out the $y$ data:

In [ ]:
y_data.head()

### (3.5): Splitting along training/validation/testing:

In [ ]:
x_remaining, x_testing, y_remaining, y_testing = train_test_split(
    x_data, y_data,
    test_size = 0.1, shuffle = True)

x_training, x_validation, y_training, y_validation = train_test_split(
    x_remaining, y_remaining,
    test_size = 0.1, shuffle = True)

In [ ]:
len(x_training)

In [ ]:
len(x_validation)

In [ ]:
len(x_testing)

## (4): DNN Stuff:

### (4.1): MSE Loss:

In [ ]:
class CrossSectionLoss(tf.keras.losses.Loss):
    def call(self, y_true, y_pred):
        return tf.reduce_mean(tf.square(y_true - y_pred))

### (4.2): DNN Architecture:

In [ ]:
class CrossSectionSurrogateModel(tf.keras.Model):

    # https://keras.io/api/models/model/ -> follow this for custom model architecture

    def __init__(self, symmetry_loss_weight = 1.0):
        super().__init__()

        self.symmetry_loss_weight = symmetry_loss_weight

        # self.data_loss_tracker = tf.keras.metrics.Mean(name = "data_loss")
        # self.symmetry_loss_tracker = tf.keras.metrics.Mean(name = "symmetry_loss")

        initializer = tf.keras.initializers.GlorotNormal(seed = None)

        self.dense_layer_1 = tf.keras.layers.Dense(32, kernel_initializer = initializer, activation = "tanh")
        self.dense_layer_2 = tf.keras.layers.Dense(32, kernel_initializer = initializer, activation = "tanh")
        self.dense_layer_3 = tf.keras.layers.Dense(32, kernel_initializer = initializer, activation = "tanh")

        # linear activation is default activation if `activation` key is not specified: https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dense
        self.cross_section_output = tf.keras.layers.Dense(1, activation = "linear", name = "cross_section")

        # custom loss business:
        self.cross_section_loss_tracker = CrossSectionLoss()

    def azimuthal_symmetry_loss(self, X_batch, training = True):

        X_plus = X_batch

        phi = X_batch[:, -1]

        X_minus = tf.concat([X_batch[:, :-1], tf.expand_dims(-phi, axis = 1)], axis = 1)

        y_plus = self(X_plus, training = training)
        y_minus = self(X_minus, training = training)

        return tf.reduce_mean(tf.square(y_plus - y_minus))

    def call(self, inputs, training = False):

        # hidden layer computation:
        hidden_layer = self.dense_layer_1(inputs)
        hidden_layer = self.dense_layer_2(hidden_layer)
        hidden_layer = self.dense_layer_3(hidden_layer)
        cross_section_output = self.cross_section_output(hidden_layer)

        return cross_section_output
    
    def train_step(self, data):

        # unpack data:
        X_batch_data, y_batch_data = data

        with tf.GradientTape() as tape:
            # forward pass:
            predictions = self(X_batch_data, training = True)

            # recall: `Instead, use `model.compute_loss(x, y, y_pred, sample_weight)`
            data_loss = self.compute_loss(X_batch_data, y_batch_data, predictions)

            # compute phi symmetry loss:
            symmetry_loss = self.azimuthal_symmetry_loss(X_batch_data, training = True)

            # total loss is just a weighted sum:
            total_loss = data_loss + self.symmetry_loss_weight * symmetry_loss

        gradients = tape.gradient(total_loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(gradients,self.trainable_variables))

        for metric in self.metrics:
            if metric.name == "loss":
                metric.update_state(total_loss)
            else:
                metric.update_state(y_batch_data, predictions)

        return {
            "loss": total_loss,
            "data_loss": data_loss,
            "symmetry_loss": symmetry_loss,
            **{m.name: m.result() for m in self.metrics}
        }

    def test_step(self, data):
        # unpack data:
        X_batch_data, y_batch_data = data
        
        # forward pass evaluation:
        predictions = self(X_batch_data, training = False)

        # recall: `Instead, use `model.compute_loss(x, y, y_pred, sample_weight)`
        data_loss = self.compute_loss(X_batch_data, y_batch_data, predictions)

         # compute phi symmetry loss:
        symmetry_loss = self.azimuthal_symmetry_loss(X_batch_data, training = False)

        # total loss is just a weighted sum:
        total_loss = data_loss + self.symmetry_loss_weight * symmetry_loss

        for metric in self.metrics:
            if metric.name == "loss":
                metric.update_state(total_loss)
            else:
                metric.update_state(y_batch_data, predictions)
            
        return {
            "loss": total_loss,
            "data_loss": data_loss,
            "symmetry_loss": symmetry_loss,
            **{m.name: m.result() for m in self.metrics}
        }

## (5): **Actually Fitting the Model**:

In [ ]:
number_of_replicas = 10

all_histories = []
all_point_predictions = []
all_smooth_predictions = []

models = []

phi_smooth = np.linspace(-1.0*np.pi, 1.0*np.pi, 361)
t_value = x_data["t"].mean()
xb_value = x_data["x_b"].mean()
qsquared_value = x_data["q_squared"].mean()
x_smooth = np.column_stack([
    np.full_like(phi_smooth, t_value),
    np.full_like(phi_smooth, xb_value),
    np.full_like(phi_smooth, qsquared_value),
    phi_smooth
])

for index in range(number_of_replicas):
    replica_number = index + 1
    print(f"[INFO]: Now training replica #{replica_number}")

    tf.keras.backend.clear_session()
    gc.collect()

    dnn_model = CrossSectionSurrogateModel(symmetry_loss_weight = 0.0)
    dnn_model.compile(
        optimizer = tf.keras.optimizers.Adam(3e-4),
        loss = CrossSectionLoss())

    dnn_model_history = dnn_model.fit(
        x_training, y_training,
        validation_data = (x_validation, y_validation),
        epochs = 750, 
        # [NOTE]: BATCHSIZE really matters!
        # batch_size = len(x_training),
        batch_size = 8,
        callbacks = [
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor = "val_loss", factor = 0.5, patience = 50, min_lr = 1e-6,
                verbose = 1),
            tf.keras.callbacks.EarlyStopping(
                monitor = "val_loss", patience = 100, restore_best_weights = True
            )
        ],
        verbose = 0)
    
    all_histories.append(dnn_model_history.history)

    model_testing_evaluation_metrics = dnn_model.evaluate(x_testing, y_testing, verbose = 0)
    print(f"[INFO]: Evaluation metrics are: {model_testing_evaluation_metrics}")

    dictionary_of_keras_metrics = dict(zip(dnn_model.metrics_names, model_testing_evaluation_metrics))
    model_testing_loss = dictionary_of_keras_metrics["loss"]
    print(f"[INFO]: Test loss for replica #{replica_number}: {model_testing_loss}")

    figure, axis = plt.subplots(1, 1, figsize = (6, 6))

    axis.plot(dnn_model_history.history['loss'], 
        label = "Training Loss", color = 'orange', alpha = 0.6)
    axis.plot(dnn_model_history.history['val_loss'], 
        label = "Validation Loss", color = 'purple', alpha = 0.6)

    axis.set_xlabel(r"Epoch", fontsize = 14.)
    axis.set_ylabel(r"Loss", fontsize = 14.)
    axis.set_title(
        rf"Surrogate Model (Testing = ${model_testing_loss:.3f}$)",
        fontsize = 18.)
    axis.legend(fontsize = 14.)
    axis.grid(visible = False)

    axis.text(
        0.00, -0.11,
        f"Figure rendered {datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}", 
        transform = axis.transAxes, verticalalignment = 'top',  horizontalalignment = 'left', fontsize = 9.,)

    for extension in ['png', 'eps']:
        figure.savefig(
            fname = f"./plots/xsec_surrogate_lc_replica_{replica_number}_v4.{extension}",
            facecolor = 'white',
            transparent = False)

    plt.close(figure)

    del figure
    del axis
    
    # predictions at actual datapoints
    replica_point_predictions = dnn_model.predict(x_data)
    all_point_predictions.append(replica_point_predictions)

    models.append(dnn_model)

    # predictions on smooth interpolation grid
    # replica_smooth_predictions = dnn_model.predict(x_smooth).flatten()
    # all_smooth_predictions.append(replica_smooth_predictions)

all_point_predictions = np.array(all_point_predictions)
all_smooth_predictions = np.array(all_smooth_predictions)

In [ ]:
grouped = test_dataframe.groupby(['t', 'x_b', 'q_squared'])

In [ ]:
average_prediction = np.mean(all_point_predictions, axis = 0)
standard_dev_prediction = np.std(all_point_predictions, axis = 0)

# this is trento convention: -pi to pi:
phi_smooth = np.linspace(-np.pi, np.pi, 361)
special_phis = [0, np.pi/2, -np.pi/2, np.pi]

for (t_value, xb_value, qsquared_value), group in grouped:
    print(f"Processing t = {t_value}, xb = {xb_value}, Q2 = {qsquared_value}")

    group = group.sort_values('phi')

    xsec_err = group['unp_beam_unp_target_xsec_err'].values

    indices = group.index.values

    xsec_pred = average_prediction[indices, 0]
    xsec_std = standard_dev_prediction[indices, 0]

    x_smooth = np.column_stack([
        np.full_like(phi_smooth, t_value),
        np.full_like(phi_smooth, xb_value),
        np.full_like(phi_smooth, qsquared_value),
        phi_smooth
    ])

    smooth_preds_all = np.array([ model.predict(x_smooth, verbose = 0) for model in models ])

    smooth_mean = np.mean(smooth_preds_all, axis = 0)
    smooth_std = np.std(smooth_preds_all, axis = 0)

    xsec_smooth_mean = smooth_mean[:, 0]
    xsec_smooth_std = smooth_std[:, 0]

    for phi_target in special_phis:
        phi_index = np.argmin(np.abs(phi_smooth - phi_target))
        phi_actual = phi_smooth[phi_index]
        sigma_value = xsec_smooth_std[phi_index]
        print(
            f"[INFO]: "
            f"(xb = {xb_value:.3f}, t = {t_value:.3f}, Q2 = {qsquared_value:.3f}) "
            f"cross-section 1σ uncertainty at phi = {phi_actual:.3f} rad is ±{sigma_value:.6f}"
        )

    phi = group['phi'].values
    xsec_actual = group['unp_beam_unp_target_xsec'].values

    xsec_res = xsec_actual - xsec_pred
    chi2_xsec = np.sum(xsec_res**2) / len(phi)

    residuals_figure, axes = plt.subplots(2, 1, figsize = (10, 8), sharex = 'col', layout = "tight")

    axes[1].text(
        -0.1, -0.1, 
        fr"Figure rendered {datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}", 
        transform = axes[1].transAxes)

    axes[0].plot(phi_smooth, xsec_smooth_mean, color = 'red', lw = 2, label = rf'Replica Average ($N = {number_of_replicas}$)')
    axes[0].fill_between(
        phi_smooth, xsec_smooth_mean - xsec_smooth_std, xsec_smooth_mean + xsec_smooth_std,
        color = 'red', alpha = 0.3,
        label = r'$\sigma$ band')
    
    axes[0].errorbar(
        phi, xsec_actual, yerr = xsec_err, 
        fmt = 'o', mfc = 'white', mec = 'black', ms = 5, ecolor = 'black', elinewidth = 1, capsize = 2, alpha = 0.8,
        label = 'Experimental Data')
    axes[0].set_ylabel(r"$d^{4}\sigma$  [nb / GeV$^{4}$]", fontsize = 14)
    axes[0].set_title(rf"Cross Section ($\chi^2_\nu = {chi2_xsec:.3f}$)")
    axes[0].legend(fontsize = 14.)
    axes[0].grid(True, linestyle = ':', alpha = 0.6)

    axes[1].scatter(phi, xsec_res, color = 'blue', alpha = 0.6)
    axes[1].axhline(0, color = 'black', linestyle = '--')
    axes[1].set_title("Residuals")
    axes[1].grid(True, linestyle = ':', alpha = 0.6)
    
    residuals_figure.suptitle(
        "Kinematic Setting:\n"
        rf"$t = {t_value:.3f}$, $x_\textrm{{B}} = {xb_value:.3f}$, $Q^2 = {qsquared_value:.3f}$",
        fontsize = 16
    )

    filename = f"./plots/t{t_value:.3f}_xb{xb_value:.3f}_q2{qsquared_value:.3f}_residuals_v4"
    
    for extension in ['png', 'eps']:
        residuals_figure.savefig(
            fname = f"{filename}.{extension}",
            facecolor = 'white', transparent = False)

    plt.close(residuals_figure)